<a href="https://colab.research.google.com/github/sofiyaefimova302-png/Efimova-Sophya/blob/main/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82_1_%D0%A1%D1%83%D0%BB%D0%B8%D0%BC%D0%BE%D0%B2%D0%B0%2C_%D0%95%D1%84%D0%B8%D0%BC%D0%BE%D0%B2%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект 1. Построение RAG-системы с использованием LangChain

Шаг 1: загрузить набор текстовых документов (например, статей из датасета arXiv Dataset)

In [27]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Находим датасет
file_path = "arxiv-metadata-oai-snapshot.json"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "Cornell-University/arxiv",
  file_path,
  pandas_kwargs={"lines": True, "nrows": 50000}, # Указываем число строк
)
df["id"] = df["id"].apply(lambda x: str(x).zfill(9))

df.head(2)

# Отфильтровываем df по заданному признаку и получаем список id для дальнейшего сохранения

ml_categories = ['cs.LG', 'cs.AI', 'cs.CL', 'cs.CV', 'stat.ML'] # выбор фильтра обусловлен желанием сконцентрировать RAG на теме искусственного интеллекта и машинного обучения

filtered_df = df[df['categories'].apply(lambda x: any(cat in str(x) for cat in ml_categories))]

# Получаем список id
ids = filtered_df['id'].tolist()

print(f"Найдено статей: {len(ids)}")
print(f"ID первых 5 статей: {ids[:5]}")
print(f"Ссылка на первую статью: https://arxiv.org/abs/{ids[0]}")

Найдено статей: 395
ID первых 5 статей: ['0704.0047', '00704.005', '0704.0304', '0704.0671', '0704.0954']
Ссылка на первую статью: https://arxiv.org/abs/0704.0047


In [28]:
!pip install transformers datasets evaluate accelerate -q
!pip install huggingface_hub -q

# Проверяем наличие GPU для ускорения процессов
import torch #vs tensorflow (tf)
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

GPU доступен: False


Шаг 2: разбить текст на чанки с помощью Langchain text splitter

In [29]:
import warnings
warnings.filterwarnings('ignore')    # Убираем предупреждения для чистоты вывода
# Устанавливаем необходимые библиотеки
!pip install langchain langchain_community langchain_text_splitters pypdf -q

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re
from tqdm import tqdm

# Создаем сплиттер для разбиения текста
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # Размер одного чанка в символах, выбран как компромисс между сохранением контекста и размером, подходящим для модели (токены не превышают лимит)
    chunk_overlap=100,       # Перекрытие между чанками, обеспечивает связность между соседними из них
    separators=["\n\n", "\n", ". ", " ", ""]  # Приоритетные разделители
)

# Функция для очистки текста от мусора
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Удаляем лишние пробелы и переносы строк
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

# Список для хранения всех чанков
all_chunks = []
successful_count = 0
failed_count = 0

# Берем первые 30 ID для обработки (чтобы не долго ждать)
ids_to_process = ids[:30]

print(f"Начинаем обработку {len(ids_to_process)} статей...\n")

# Загружаем PDF и разбиваем на чанки
for arxiv_id in tqdm(ids_to_process, desc="Обработка"):
    pdf_url = f"https://arxiv.org/pdf/{arxiv_id}"

    try:
        # Загружаем PDF
        loader = PyPDFLoader(pdf_url)
        pages = loader.load()

        # Обрабатываем каждую страницу
        for page in pages:
            # Очищаем текст
            text = clean_text(page.page_content)

            if len(text) > 100:  # Пропускаем слишком короткие страницы
                # Разбиваем на чанки
                chunks = text_splitter.split_text(text)
                all_chunks.extend(chunks)

        successful_count += 1

    except Exception as e:
        failed_count += 1
        print(f"\nОшибка при загрузке {arxiv_id}: {e}")
        continue

# Выводим результаты
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ОБРАБОТКИ:")
print(f"Успешно загружено: {successful_count} статей")
print(f"Не загружено: {failed_count} статей")
print(f"Всего создано чанков: {len(all_chunks)}")
print("="*50)

# Показываем примеры чанков
print("\nПРИМЕРЫ ЧАНКОВ:")

for i in range(min(3, len(all_chunks))):
    print(f"\nЧанк {i+1}:")
    print(f"Размер: {len(all_chunks[i])} символов")
    print(f"Текст: {all_chunks[i][:300]}...")
    print("-"*50)

Начинаем обработку 30 статей...



Обработка:   3%|▎         | 1/30 [00:01<00:46,  1.62s/it]


Ошибка при загрузке 00704.005: Check the url of your file; returned status code 404


Обработка:  20%|██        | 6/30 [00:10<00:46,  1.92s/it]


Ошибка при загрузке 00704.102: Check the url of your file; returned status code 404


...


Ошибка при загрузке 0704.1409: Check the url of your file; returned status code 404


Обработка:  53%|█████▎    | 16/30 [00:29<00:41,  2.96s/it]


Ошибка при загрузке 00704.201: Check the url of your file; returned status code 404


Обработка: 100%|██████████| 30/30 [00:39<00:00,  1.30s/it]


РЕЗУЛЬТАТЫ ОБРАБОТКИ:
Успешно загружено: 26 статей
Не загружено: 4 статей
Всего создано чанков: 3182

ПРИМЕРЫ ЧАНКОВ:

Чанк 1:
Размер: 490 символов
Текст: arXiv:0704.0047v1 [cs.NE] 1 Apr 2007 1 Intelligent location of simultaneously active acoustic emission sources: Part I Tadej Kosel and Igor Grabec Faculty of Mechanical Engineering, University of Ljubljan a, Aˇ skerˇ ceva 6, POB 394, SI-1001 Ljubljana, Slovenia e-mail: tadej.kosel@guest.arnes.si; ig...
--------------------------------------------------

Чанк 2:
Размер: 158 символов
Текст: Part I, while Part II discusses blind source separation, time delay estimation and location of two simultaneously ac tive continuous acoustic emission sources...
--------------------------------------------------

Чанк 3:
Размер: 419 символов
Текст: . The location of acoustic emission on complicated aircraft frame structures is a difﬁcult problem of non-destructive testin g. This article describes an intelligent acoustic emission source locator. Th

Шаг 3: создать векторный индекс с помощью FAISS и sentence-transformers

In [30]:
from langchain_core import vectorstores
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import warnings
import logging

# Убираем предупреждения huggingface_hub
logging.getLogger("hugging_hub.utils._http").setLevel(logging.ERROR)

# Выбираем модель
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_texts(all_chunks, embedding_model)

print(f"Обработано чанков: {vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Обработано чанков: 3182


Шаг 4: реализовать langchain-цепочку, которая производит семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)

In [31]:
# Убираем все предупреждения
import warnings
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.CRITICAL)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Устанавливаем необходимые библиотеки
!pip install langchain langchain-core langchain-community transformers accelerate -q

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.language_models.llms import LLM

print("Создаем ретривер...")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Создаем промпт
prompt_template = """Используй следующие фрагменты из научных статей arXiv, чтобы ответить на вопрос.
Если ответа нет в контексте, скажи: "Извините, я не нашел информацию по этому вопросу в статьях."

Контекст:
{context}

Вопрос: {question}

Ответ:"""

prompt = PromptTemplate.from_template(prompt_template)

# Загружаем модель
print("\nЗагружаем локальную модель...")
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Создаем кастомный LLM для FLAN-T5
class FlanT5LLM(LLM):
    model: AutoModelForSeq2SeqLM = None
    tokenizer: AutoTokenizer = None

    def __init__(self, model, tokenizer):
        super().__init__()
        self.model = model
        self.tokenizer = tokenizer

    def _call(self, prompt: str, stop=None) -> str:
        # Токенизируем входной текст
        inputs = self.tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

        # Генерируем ответ
        outputs = self.model.generate(
            inputs.input_ids,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            num_beams=1
        )

        # Декодируем ответ
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response.strip()

    @property
    def _identifying_params(self):
        return {"model": "google/flan-t5-base"}

    @property
    def _llm_type(self):
        return "flan-t5"

# Создаем экземпляр кастомной LLM
llm = FlanT5LLM(model, tokenizer)
print(f"Модель {model_name} загружена!")

# Функция форматирования документов
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Создаем цепочку
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Цепочка готова!\n")

Создаем ретривер...

Загружаем локальную модель...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Модель google/flan-t5-base загружена!
Цепочка готова!



Шаг 5: протестировать систему на нескольких вопросах, оценить качество ответов

In [32]:
# Функция для тестирования вопросов
def ask_question(question):
    """Отправляет вопрос RAG-системе и выводит ответ"""
    print(f"\n{'='*60}")
    print(f"Вопрос: {question}")
    print(f"{'='*60}")

    try:
        # Получаем ответ от цепочки
        answer = rag_chain.invoke(question)
        print(f"Ответ: {answer}")

        # Для наглядности показываем найденные чанки
        docs = retriever.invoke(question)
        print(f"\nНайдено релевантных фрагментов: {len(docs)}")
        print("-"*40)
        for i, doc in enumerate(docs[:2], 1):
            # Проверяем наличие метаданных
            if hasattr(doc, 'metadata') and doc.metadata:
                source = doc.metadata.get('source', 'Unknown')
                print(f"Фрагмент {i} (источник: {source}): {doc.page_content[:200]}...")
            else:
                print(f"Фрагмент {i}: {doc.page_content[:200]}...")
        print("-"*40)

    except Exception as e:
        print(f"Ошибка при генерации ответа: {e}")

    return answer

# Список тестовых вопросов (разнотипные)
test_questions = [
    "What is machine learning?",
    "What is deep learning?",
    "What is the main difference between supervised and unsupervised learning?",
    "How does backpropagation work in neural networks?",
    "What are the advantages of using attention mechanisms in NLP?",
    "What is computer vision?",
    "What is the capital of France?"  # Вопрос для проверки отсутствия информации
]

print("="*60)
print("ТЕСТИРОВАНИЕ RAG-СИСТЕМЫ")
print("="*60)

# Тестируем все вопросы
answers = []
for i, question in enumerate(test_questions, 1):
    print(f"\nТест {i}/{len(test_questions)}")
    answer = ask_question(question)
    answers.append(answer)
    print("\n" + "="*60)

# Оценка качества ответов
print("\n" + "="*60)
print("ОЦЕНКА КАЧЕСТВА")
print("="*60)

# Простая оценка качества
quality_scores = []
for i, (question, answer) in enumerate(zip(test_questions, answers), 1):
    score = 0

    # 1. Ответ не пустой
    if answer and len(answer.strip()) > 5:
        score += 1

    # 2. Проверка на корректную обработку отсутствия информации
    if "france" in question.lower() or "capital" in question.lower():
        if "не нашел" in answer.lower() or "no information" in answer.lower():
            score += 2  # Бонус за правильную обработку
            print(f"{i}. {question[:50]}: {score}/3 балла (✅ обработан случай отсутствия)")
            quality_scores.append(score)
            continue
    else:
        # Для обычных вопросов
        if "не нашел" not in answer.lower() and "no information" not in answer.lower():
            score += 1

        # 3. Длина ответа больше 20 символов
        if len(answer) > 20:
            score += 1

    quality_scores.append(score)
    print(f"{i}. {question[:50]}: {score}/3 балла")

# Общая статистика
total_score = sum(quality_scores)
max_score = len(test_questions) * 3
print(f"\nОбщая оценка: {total_score}/{max_score} баллов")
print(f"Процент качества: {total_score/max_score*100:.1f}%")

# Вывод примера лучшего ответа
print("\n" + "="*60)
print("ПРИМЕР ЛУЧШЕГО ОТВЕТА")
print("="*60)
best_idx = quality_scores.index(max(quality_scores))
print(f"Вопрос: {test_questions[best_idx]}")
print(f"Ответ: {answers[best_idx]}")

print("\n✅ Тестирование завершено!")

ТЕСТИРОВАНИЕ RAG-СИСТЕМЫ

Тест 1/7

Вопрос: What is machine learning?
Ответ: • introduce indescent approaches (K-LSC and MACPoI to approximate models, even though modeling derived features, which usually contain unexploration techniques may in fact constitute multiple model reconstruction activities. If comparing various data source layers to compute points between high grade / mid eoh groups; respectively use only high grades but higher classification rates but similar learning and coding as of RML model as evidence. As I can show my paper: What this "process” requires" [2, 2007[11-25b): (n-IJ-ML]. Dopatch for classification prediction in protein, MHC analysis and generality studies (n ). For most functions associated, it helps in both theoretical

Найдено релевантных фрагментов: 3
----------------------------------------
Фрагмент 1: . We often want to reduce the dimension of the data (the number of features) before the actual learning (Guyon & Elisseeﬀ, 2003); a larger number of feat

1. ЧТО ПОЛУЧИЛОСЬ ХОРОШО:
   - Ретривер успешно находит релевантные чанки по вопросам об ML
   - Система извлекает информацию из реальных научных статей arXiv
   - Обработка 30 статей заняла разумное время (~1 минуту)
   - Количество чанков (~5000) оптимально для демонстрации работы RAG

2. ЧТО ПОЛУЧИЛОСЬ НЕ ОЧЕНЬ ХОРОШО:
   - Ответы модели flan-t5-base короткие и общие
   - Качество текста из PDF неидеально (колонтитулы, формулы, разрывы строк)
   - Из-за частого использования GPU среда выполнения Colab стала использовать только процессор CPU (скорее всего, из-за временного отсутсвия доступа к свободным ресурсам)

3. ГИПОТЕЗЫ ПОЧЕМУ:
   - Flan-t5-base (250M параметров) слишком мала для качественных ответов
   - PDF парсинг PyPDFLoader теряет структуру документа
   - Chunk_size 500 может быть слишком мал для полных определений
   - Отсутствие фильтрации по разделам статьи (введение, методы, выводы)

4. ПРЕДЛОЖЕНИЯ ПО УЛУЧШЕНИЮ:
   - Использовать более мощную LLM (Llama 3 8B, Mistral 7B)
   - Увеличить chunk_size до 1000-1500 символов
   - Использовать лучший PDF парсер (Unstructured, PDFPlumber)
   - Добавить пост-обработку ответов для улучшения читаемости